# Step2: Annotation, Composition & Differential Analysis — 202507LPJ

Audit-grade Step2 workflow for the LPJ BBB-related chemotherapy + light/RT project.

This notebook covers:

1. Load Step2 preprocessed AnnData from Step1.
2. Cluster review and cluster marker discovery.
3. Marker/enrichment-based annotation evidence generation.
4. Manual annotation mapping and final label QC.
5. BBB function gene-set scoring and visualization.
6. Cell composition comparison between PBS and RT.
7. Celltype-aware PBS vs RT differential expression and enrichment.
8. BBB-focused marker/signature visualization and export.

The notebook is intentionally modular: rerun annotation, composition, or DE sections independently after updating manual mappings.


## 1. Environment Setup


In [ ]:
from pathlib import Path
import gc
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

import scLucid as scl

warnings.filterwarnings("ignore")
scl.setup_logging("INFO")
scl.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 5),
    style="seaborn-v0_8",
    style_dict={"axes.grid": True},
    color_theme="default",
)
sc.settings.verbosity = 2

## 2. Project Configuration


In [ ]:
BASE_DIR = Path('/Users/luye/Library/Mobile Documents/com~apple~CloudDocs/Projects/Ongoing/202507LPJ')
DATA_DIR = BASE_DIR / "1-DATA"
RESULTS_DIR = BASE_DIR / "2-OUTPUT"

INPUT_H5AD = DATA_DIR / "Step2-sce_preprocessed.h5ad"
ANNOTATED_H5AD = DATA_DIR / "Step3-sce_annotated_scLucid_LPJ.h5ad"
REVIEW_DIR = RESULTS_DIR / "step2_scLucid_annotation"
COMPOSITION_DIR = RESULTS_DIR / "step2_composition"
DE_DIR = RESULTS_DIR / "step2_celltype_de"
FIG_DIR = RESULTS_DIR / "step2_figures"

for d in [REVIEW_DIR, COMPOSITION_DIR, DE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY = "sampleID"
GROUP_KEY = "group"
GROUP_ORDER = ["PBS", "RT"]
CONDITION_1 = "PBS"
CONDITION_2 = "RT"
SPECIES = "mouse"

# Use the integrated representation if available, otherwise fall back gracefully.
PREFERRED_REP_KEYS = ["X_harmony", "X_pca"]
PREFERRED_UMAP_KEYS = ["X_umap_harmony", "X_umap_pca", "X_umap"]

CLUSTER_KEY = "leiden_clusters"
MERGED_CLUSTER_KEY = "leiden_clusters_merged"
ANNOTATION_KEY = "celltype"
MAIN_ANNOTATION_KEY = "main_celltype"

CLUSTER_RESOLUTION = 0.8
MERGE_SIMILARITY_THRESHOLD = 0.4

# Manual mapping file generated/reused by the review workflow.
MAPPING_FILE = REVIEW_DIR / "manual_annotation_mapping.xlsx"

SAMPLE_COLORS = {
    "GSGC0384234": "#1E688D",
    "GSGC0384235": "#4DB3E6",
    "GSGC0384236": "#8B4C9D",
    "GSGC0384237": "#C77CFF",
}
GROUP_COLORS = {"PBS": "#3A8AE6", "RT": "#D45087"}

print(f"Input: {INPUT_H5AD}")
print(f"Outputs: {REVIEW_DIR}")

## 3. Load Data & Basic Audit


In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"obs columns: {list(adata.obs.columns)[:30]}")
print(f"obsm keys: {list(adata.obsm.keys())}")

if GROUP_KEY not in adata.obs.columns:
    group_dict = {
        "GSGC0384234": "PBS",
        "GSGC0384235": "PBS",
        "GSGC0384236": "RT",
        "GSGC0384237": "RT",
    }
    adata.obs[GROUP_KEY] = adata.obs[SAMPLE_KEY].map(group_dict).fillna("Unknown")

adata.obs[GROUP_KEY] = pd.Categorical(adata.obs[GROUP_KEY], categories=GROUP_ORDER, ordered=True)

USE_REP = next((k for k in PREFERRED_REP_KEYS if k in adata.obsm), None)
UMAP_KEY = next((k for k in PREFERRED_UMAP_KEYS if k in adata.obsm), None)
if USE_REP is None:
    raise KeyError(f"None of preferred representation keys found: {PREFERRED_REP_KEYS}")
if UMAP_KEY is None:
    raise KeyError(f"None of preferred UMAP keys found: {PREFERRED_UMAP_KEYS}")

print(f"Using representation: {USE_REP}")
print(f"Using UMAP basis: {UMAP_KEY}")
print(pd.crosstab(adata.obs[SAMPLE_KEY], adata.obs[GROUP_KEY], margins=True))

In [ ]:
palette = scl.pl.build_obs_palette(
    adata,
    [SAMPLE_KEY, GROUP_KEY],
    color_maps={"samples": SAMPLE_COLORS, "groups": GROUP_COLORS},
)
sc.pl.embedding(
    adata,
    basis=UMAP_KEY,
    color=[GROUP_KEY, SAMPLE_KEY],
    palette=palette,
    ncols=2,
    frameon=False,
    show=True,
)

## 4. Clustering Review


In [ ]:
cluster_cfg = scl.al.ClusteringConfig(
    method="leiden",
    resolution=CLUSTER_RESOLUTION,
    use_rep=USE_REP,
    key_added=CLUSTER_KEY,
    random_state=42,
    plot=False,
    save_dir=str(REVIEW_DIR / "clustering"),
)
adata = scl.al.cluster_cells(adata, config=cluster_cfg)

adata = scl.al.merge_clusters(
    adata,
    cluster_key=CLUSTER_KEY,
    similarity_threshold=MERGE_SIMILARITY_THRESHOLD,
)

# Keep a stable alias for downstream sections.
if MERGED_CLUSTER_KEY not in adata.obs.columns and f"{CLUSTER_KEY}_merged" in adata.obs.columns:
    MERGED_CLUSTER_KEY = f"{CLUSTER_KEY}_merged"

print(adata.obs[CLUSTER_KEY].value_counts().sort_index())
if MERGED_CLUSTER_KEY in adata.obs.columns:
    print(adata.obs[MERGED_CLUSTER_KEY].value_counts().sort_index())

In [ ]:
sc.pl.embedding(
    adata,
    basis=UMAP_KEY,
    color=[CLUSTER_KEY, MERGED_CLUSTER_KEY if MERGED_CLUSTER_KEY in adata.obs.columns else CLUSTER_KEY],
    legend_loc="on data",
    legend_fontsize=7,
    ncols=2,
    frameon=False,
    show=True,
)

## 5. Cluster Markers & Enrichment Evidence


In [ ]:
MARKER_GROUPBY = MERGED_CLUSTER_KEY if MERGED_CLUSTER_KEY in adata.obs.columns else CLUSTER_KEY

marker_cfg = scl.al.DifferentialConfig(
    groupby=MARKER_GROUPBY,
    method="wilcoxon",
    use_raw=True,
    key_added="rank_genes_groups",
    pval_cutoff=0.05,
)
scl.al.find_markers(adata, config=marker_cfg)

filter_cfg = scl.al.FilterMarkersConfig(
    key="rank_genes_groups",
    key_added="highly_specific_markers_df",
    min_log2fc=1.0,
    max_padj=0.01,
    min_in_group_pct=0.2,
    max_out_group_pct=None,
    min_diff_pct=0.1,
    keep_top_n=50,
)
highly_specific_markers_df = scl.al.filter_markers(adata, config=filter_cfg)
highly_specific_markers_df.to_csv(REVIEW_DIR / "highly_specific_cluster_markers.csv", index=False)
display(highly_specific_markers_df.head(20))

In [ ]:
try:
    scl.al.visualize_markers(
        adata,
        markers=highly_specific_markers_df,
        groupby=MARKER_GROUPBY,
        plot_type="dotplot",
        n_genes_per_group=5,
    )
except Exception as exc:
    print(f"Marker dotplot skipped: {exc}")

try:
    enr_cfg = scl.al.EnrichmentConfig(
        de_key="rank_genes_groups_df",
        mode="offline",
        organism=SPECIES,
        gene_sets_offline=["go_bp", "reactome"],
    )
    scl.al.run_enrichment(adata, groupby=MARKER_GROUPBY, config=enr_cfg)
    scl.al.summarize_markers_and_enrichment(
        adata,
        groupby=MARKER_GROUPBY,
        markers_df=highly_specific_markers_df,
        enrichment_key="enrichment",
        n_markers=25,
        n_terms=10,
        summary_file=str(REVIEW_DIR / "annotation_marker_enrichment_summary.md"),
    )
except Exception as exc:
    print(f"Offline enrichment skipped: {exc}")

## 6. BBB-Focused Annotation Evidence

These marker panels are review aids, not final labels. Edit the marker lists for LPJ-specific biology as needed.


In [ ]:
BBB_MARKER_PANELS = {
    "Endothelial_BBB": ["Pecam1", "Cdh5", "Kdr", "Flt1", "Cldn5", "Ocln", "Tjp1", "Slc2a1", "Abcb1a", "Mfsd2a", "Plvap"],
    "Pericyte_VSMC": ["Pdgfrb", "Rgs5", "Kcnj8", "Abcc9", "Acta2", "Tagln", "Myh11", "Cspg4"],
    "Astrocyte": ["Aqp4", "Gfap", "Aldh1l1", "Slc1a2", "Slc1a3", "S100b"],
    "Microglia_Macrophage": ["P2ry12", "Cx3cr1", "Tmem119", "Aif1", "Lyz2", "C1qa", "C1qb", "Tyrobp"],
    "Oligodendrocyte_Lineage": ["Pdgfra", "Cspg4", "Mbp", "Mog", "Plp1", "Sox10"],
    "Neuron": ["Snap25", "Syt1", "Rbfox3", "Tubb3", "Slc17a7", "Gad1", "Gad2"],
    "T_NK": ["Ptprc", "Cd3d", "Cd3e", "Nkg7", "Gzma", "Trac"],
    "B_Plasma": ["Ms4a1", "Cd79a", "Cd74", "Jchain", "Mzb1"],
    "Proliferation": ["Mki67", "Top2a", "Pcna", "Tyms", "Rrm2"],
    "Stress_Inflammation": ["Fos", "Jun", "Hspa1a", "Hspa1b", "Ifit1", "Isg15", "Cxcl10", "Il1b", "Tnf"],
}

def present_genes(adata, genes):
    var_names = set(map(str, adata.var_names))
    raw_names = set(map(str, adata.raw.var_names)) if adata.raw is not None else set()
    return [g for g in genes if g in var_names or g in raw_names]

panel_hits = {panel: present_genes(adata, genes) for panel, genes in BBB_MARKER_PANELS.items()}
pd.Series({k: len(v) for k, v in panel_hits.items()}, name="n_present").to_csv(REVIEW_DIR / "bbb_marker_panel_coverage.csv")
panel_hits

In [ ]:
for panel, genes in panel_hits.items():
    if not genes:
        continue
    genes_to_plot = genes[:8]
    print(f"Plotting {panel}: {genes_to_plot}")
    sc.pl.embedding(
        adata,
        basis=UMAP_KEY,
        color=genes_to_plot,
        use_raw=True,
        cmap="Reds",
        vmin="p1",
        vmax="p99",
        ncols=4,
        frameon=False,
        show=True,
    )

## 6B. BBB Function Gene-Set Scoring

Functional signature scoring adapted from the LVA hippocampus workflow. The LPJ version keeps BBB tight junction, endothelial transport/leakage, hyaluronan/ECM remodeling, mural-cell support, astrocyte endfeet, and inflammatory BBB stress as explicit axes.


In [ ]:
# BBB/vascular function signatures for LPJ
# These sets are used for module scoring and visual QC. Missing genes are filtered automatically.
USE_RAW_FOR_SIGNATURES = adata.raw is not None
SIGNATURE_GROUPBY = ANNOTATION_KEY if ANNOTATION_KEY in adata.obs.columns else MARKER_GROUPBY

BBB_FUNCTION_GENE_SETS = {
    "BBB_Tight_Junction": ["Cldn5", "Ocln", "Tjp1", "Tjp2", "F11r", "Esam", "Cdh5", "Jam2", "Jam3", "Mfsd2a"],
    "BBB_Transport_Efflux": ["Slc2a1", "Slc7a5", "Slc16a1", "Tfrc", "Mfsd2a", "Abcb1a", "Abcb1b", "Abcg2", "Abcc1", "Lrp1"],
    "Fenestration_Transcytosis": ["Plvap", "Cav1", "Cav2", "Cavin1", "Ackr1", "Vwf"],
    "Endothelial_Activation_Adhesion": ["Icam1", "Vcam1", "Sele", "Selp", "Cxcl1", "Cxcl2", "Ccl2", "Pecam1"],
    "Angiogenesis_Shear_Response": ["Klf2", "Klf4", "Piezo1", "Nos3", "Sox17", "Esm1", "Angpt2", "Apln", "Vegfa", "Mdk"],
    "Hyaluronan_Synthesis": ["Has1", "Has2", "Has3", "Ugdh", "Ugp2", "Gfpt1", "Gfpt2"],
    "Hyaluronan_Receptor_Degradation": ["Cd44", "Hmmr", "Lyve1", "Stab2", "Hyal1", "Hyal2", "Cemip", "Tmem2"],
    "Basement_Membrane_ECM": ["Col4a1", "Col4a2", "Col18a1", "Lama4", "Lamb1", "Nid1", "Hspg2", "Fn1"],
    "Pericyte_Mural_Support": ["Pdgfrb", "Rgs5", "Cspg4", "Abcc9", "Kcnj8", "Acta2", "Myh11", "Tagln", "Notch3"],
    "Astrocyte_Endfeet_BBB": ["Aqp4", "Gja1", "Slc1a2", "Slc1a3", "Aldh1l1", "Gfap", "Apoe", "Clu"],
    "Inflammatory_BBB_Stress": ["Tnf", "Il1b", "C3", "Nfkbia", "Socs3", "Serpine1", "Hif1a", "Ptgs2"],
}

BBB_AXIS_GENE_PANELS = {
    "Tight junction / BBB integrity": BBB_FUNCTION_GENE_SETS["BBB_Tight_Junction"],
    "Transport / efflux": BBB_FUNCTION_GENE_SETS["BBB_Transport_Efflux"],
    "Fenestration / leakage": BBB_FUNCTION_GENE_SETS["Fenestration_Transcytosis"],
    "Hyaluronan / ECM remodeling": sorted(set(BBB_FUNCTION_GENE_SETS["Hyaluronan_Synthesis"] + BBB_FUNCTION_GENE_SETS["Hyaluronan_Receptor_Degradation"] + BBB_FUNCTION_GENE_SETS["Basement_Membrane_ECM"])),
    "Mural / astrocyte BBB support": sorted(set(BBB_FUNCTION_GENE_SETS["Pericyte_Mural_Support"] + BBB_FUNCTION_GENE_SETS["Astrocyte_Endfeet_BBB"])),
}


def _signature_var_names(adata, use_raw=True):
    if use_raw and adata.raw is not None:
        return pd.Index(adata.raw.var_names)
    return pd.Index(adata.var_names)


def _genes_present(adata, genes, use_raw=True):
    var_names = set(_signature_var_names(adata, use_raw=use_raw))
    return [g for g in genes if g in var_names]


def score_bbb_function_gene_sets(adata, gene_sets, use_raw=True, prefix="score_BBB"):
    records = []
    score_cols = []
    for name, genes in gene_sets.items():
        genes_in = _genes_present(adata, genes, use_raw=use_raw)
        records.append({
            "signature": name,
            "n_genes_defined": len(genes),
            "n_genes_detected": len(genes_in),
            "genes_detected": ",".join(genes_in),
            "genes_missing": ",".join([g for g in genes if g not in genes_in]),
        })
        if len(genes_in) < 2:
            print(f"Skipping {name}: only {len(genes_in)} detected gene(s).")
            continue
        score_col = f"{prefix}_{name}"
        if score_col not in adata.obs.columns:
            sc.tl.score_genes(
                adata,
                gene_list=genes_in,
                score_name=score_col,
                use_raw=use_raw,
                ctrl_size=min(50, max(1, len(_signature_var_names(adata, use_raw=use_raw)) - 1)),
            )
        score_cols.append(score_col)
    detected_df = pd.DataFrame(records)
    return score_cols, detected_df

score_cols, bbb_signature_gene_audit = score_bbb_function_gene_sets(
    adata,
    BBB_FUNCTION_GENE_SETS,
    use_raw=USE_RAW_FOR_SIGNATURES,
)
bbb_signature_gene_audit.to_csv(REVIEW_DIR / "bbb_function_signature_gene_audit.csv", index=False)
display(bbb_signature_gene_audit)
print(f"Computed {len(score_cols)} BBB function signature scores using groupby={SIGNATURE_GROUPBY!r}.")


In [ ]:
def _save_current_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_bbb_signature_mean_heatmap(adata, score_cols, groupby, save_path):
    if not score_cols:
        print("No signature scores available for heatmap.")
        return None
    mean_df = adata.obs[[groupby] + score_cols].copy().groupby(groupby, observed=True).mean()
    mean_df.columns = [c.replace("score_BBB_", "") for c in mean_df.columns]
    scaled = mean_df.apply(lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) else 1), axis=0)
    height = max(5, 0.35 * scaled.shape[0])
    width = max(8, 0.5 * scaled.shape[1])
    plt.figure(figsize=(width, height))
    sns.heatmap(scaled, cmap="RdBu_r", center=0, linewidths=0.4, linecolor="white", cbar_kws={"label": "Column z-score"})
    plt.title(f"BBB function signatures by {groupby}")
    plt.xlabel("")
    plt.ylabel("")
    plt.xticks(rotation=45, ha="right")
    _save_current_fig(save_path)
    return scaled


def plot_bbb_signature_delta_heatmap(adata, score_cols, groupby, condition_key, ref_group, target_group, save_path):
    if not score_cols:
        print("No signature scores available for delta heatmap.")
        return None
    obs = adata.obs[[groupby, condition_key] + score_cols].copy()
    ref = obs[obs[condition_key].astype(str) == str(ref_group)].groupby(groupby, observed=True)[score_cols].mean()
    tgt = obs[obs[condition_key].astype(str) == str(target_group)].groupby(groupby, observed=True)[score_cols].mean()
    common = ref.index.intersection(tgt.index)
    if len(common) == 0:
        print("No common groups between conditions for delta heatmap.")
        return None
    delta = tgt.loc[common] - ref.loc[common]
    delta.columns = [c.replace("score_BBB_", "") for c in delta.columns]
    vmax = np.nanmax(np.abs(delta.values)) if delta.size else 1
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1
    height = max(5, 0.35 * delta.shape[0])
    width = max(8, 0.5 * delta.shape[1])
    plt.figure(figsize=(width, height))
    sns.heatmap(delta, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, annot=True, fmt=".2f", linewidths=0.4, linecolor="white", cbar_kws={"label": f"Delta score ({target_group} - {ref_group})"})
    plt.title(f"BBB function remodeling by {groupby}")
    plt.xlabel("")
    plt.ylabel("")
    plt.xticks(rotation=45, ha="right")
    _save_current_fig(save_path)
    return delta


def plot_bbb_signature_group_boxplots(adata, score_cols, condition_key, save_path, max_signatures=8):
    selected = score_cols[:max_signatures]
    if not selected:
        print("No signature scores available for boxplots.")
        return None
    plot_df = adata.obs[[condition_key] + selected].melt(id_vars=condition_key, var_name="signature", value_name="score")
    plot_df["signature"] = plot_df["signature"].str.replace("score_BBB_", "", regex=False)
    plt.figure(figsize=(max(9, 1.0 * len(selected)), 4.8))
    sns.boxplot(data=plot_df, x="signature", y="score", hue=condition_key, showfliers=False, palette=GROUP_COLORS)
    plt.axhline(0, color="0.7", linewidth=0.8)
    plt.title("BBB function signature scores by condition")
    plt.xlabel("")
    plt.ylabel("Module score")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title=condition_key, bbox_to_anchor=(1.02, 1), loc="upper left")
    _save_current_fig(save_path)
    return plot_df

mean_signature_matrix = plot_bbb_signature_mean_heatmap(
    adata,
    score_cols,
    groupby=SIGNATURE_GROUPBY,
    save_path=FIG_DIR / "bbb_function_signature_mean_heatmap.pdf",
)
if mean_signature_matrix is not None:
    mean_signature_matrix.to_csv(REVIEW_DIR / "bbb_function_signature_mean_by_group.csv")

delta_signature_matrix = plot_bbb_signature_delta_heatmap(
    adata,
    score_cols,
    groupby=SIGNATURE_GROUPBY,
    condition_key=GROUP_KEY,
    ref_group=CONDITION_1,
    target_group=CONDITION_2,
    save_path=FIG_DIR / "bbb_function_signature_delta_RT_vs_PBS_heatmap.pdf",
)
if delta_signature_matrix is not None:
    delta_signature_matrix.to_csv(REVIEW_DIR / "bbb_function_signature_delta_RT_vs_PBS.csv")

boxplot_df = plot_bbb_signature_group_boxplots(
    adata,
    score_cols,
    condition_key=GROUP_KEY,
    save_path=FIG_DIR / "bbb_function_signature_scores_by_condition.pdf",
)


In [ ]:
# Gene-level BBB axis dotplots, adapted from the LVA hippocampus marker visualization.
valid_bbb_axis_gene_panels = {
    panel: _genes_present(adata, genes, use_raw=USE_RAW_FOR_SIGNATURES)
    for panel, genes in BBB_AXIS_GENE_PANELS.items()
}
valid_bbb_axis_gene_panels = {panel: genes for panel, genes in valid_bbb_axis_gene_panels.items() if genes}

if valid_bbb_axis_gene_panels:
    dotplot_obj = sc.pl.dotplot(
        adata,
        var_names=valid_bbb_axis_gene_panels,
        groupby=SIGNATURE_GROUPBY,
        use_raw=USE_RAW_FOR_SIGNATURES,
        standard_scale="var",
        cmap="RdBu_r",
        dot_max=0.8,
        dot_min=0.05,
        figsize=(12, max(4, 0.35 * adata.obs[SIGNATURE_GROUPBY].nunique())),
        show=False,
        return_fig=True,
    )
    dotplot_obj.savefig(FIG_DIR / "bbb_axis_gene_dotplot_by_annotation.pdf")
    plt.close("all")
else:
    print("No BBB axis genes were detected for dotplot.")

# Focused FeaturePlot panels for the two axes borrowed from LVA: tight junction and hyaluronan/ECM.
feature_genes = []
for genes in [
    BBB_FUNCTION_GENE_SETS["BBB_Tight_Junction"],
    BBB_FUNCTION_GENE_SETS["Hyaluronan_Synthesis"],
    BBB_FUNCTION_GENE_SETS["Hyaluronan_Receptor_Degradation"],
]:
    feature_genes.extend(_genes_present(adata, genes, use_raw=False))
feature_genes = list(dict.fromkeys(feature_genes))[:18]
if feature_genes and UMAP_KEY:
    fig = sc.pl.embedding(
        adata,
        basis=UMAP_KEY,
        color=feature_genes,
        cmap="RdBu_r",
        ncols=3,
        frameon=False,
        show=False,
        return_fig=True,
    )
    fig.savefig(FIG_DIR / "bbb_tight_junction_hyaluronan_featureplots.pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)
print("BBB function visualizations saved to", FIG_DIR)


## 7. Manual Annotation Mapping

The workflow exports a mapping template if no mapping exists. Fill `celltype` and optional `main_celltype`, then rerun this section.


In [ ]:
clusters = sorted(adata.obs[MARKER_GROUPBY].astype(str).unique(), key=lambda x: (len(x), x))
if not MAPPING_FILE.exists():
    marker_preview = (
        highly_specific_markers_df
        .assign(group=lambda d: d["group"].astype(str))
        .groupby("group")["names"]
        .apply(lambda s: ", ".join(map(str, s.head(10))))
        .reindex(clusters)
        .reset_index()
        .rename(columns={"group": "cluster", "names": "top_markers"})
    )
    marker_preview["cell_type"] = ""
    marker_preview["main_celltype"] = ""
    marker_preview["review_note"] = ""
    marker_preview.to_excel(MAPPING_FILE, index=False)
    print(f"Created manual mapping template: {MAPPING_FILE}")
else:
    print(f"Using existing mapping file: {MAPPING_FILE}")

mapping_preview = pd.read_excel(MAPPING_FILE)
display(mapping_preview.head(20))

In [ ]:
if MAPPING_FILE.exists():
    scl.al.apply_annotation_mapping(
        adata,
        cluster_key=MARKER_GROUPBY,
        mapping=str(MAPPING_FILE),
        key_added=ANNOTATION_KEY,
    )
    mapping_df = pd.read_excel(MAPPING_FILE)
    cluster_col = "cluster" if "cluster" in mapping_df.columns else MARKER_GROUPBY
    if "main_celltype" in mapping_df.columns and mapping_df["main_celltype"].notna().any():
        main_map = dict(zip(mapping_df[cluster_col].astype(str), mapping_df["main_celltype"].astype(str)))
        adata.obs[MAIN_ANNOTATION_KEY] = adata.obs[MARKER_GROUPBY].astype(str).map(main_map).fillna("Unknown").astype("category")
    else:
        adata.obs[MAIN_ANNOTATION_KEY] = adata.obs[ANNOTATION_KEY].astype("category")

print(adata.obs[ANNOTATION_KEY].value_counts(dropna=False))
print(pd.crosstab(adata.obs[ANNOTATION_KEY], adata.obs[GROUP_KEY], margins=True))

In [ ]:
palette = scl.pl.build_obs_palette(
    adata,
    [ANNOTATION_KEY, MAIN_ANNOTATION_KEY, GROUP_KEY, SAMPLE_KEY],
    color_maps={"samples": SAMPLE_COLORS, "groups": GROUP_COLORS},
)
sc.pl.embedding(
    adata,
    basis=UMAP_KEY,
    color=[ANNOTATION_KEY, MAIN_ANNOTATION_KEY, GROUP_KEY],
    palette=palette,
    legend_fontsize=8,
    ncols=3,
    frameon=False,
    show=True,
)

## 8. Cell Composition Comparison


In [ ]:
composition_key = ANNOTATION_KEY if ANNOTATION_KEY in adata.obs.columns else MARKER_GROUPBY

prop_cfg = scl.al.ProportionConfig(
    celltype_col=composition_key,
    sample_col=SAMPLE_KEY,
    condition_col=GROUP_KEY,
    test_method="clr-t-test",
    composition_transform="clr",
    require_biological_replicates=True,
    min_samples_per_condition=2,
    out_dir=str(COMPOSITION_DIR),
    export_data=True,
)
prop_df, stat_df = scl.al.analyze_celltype_proportion(
    adata,
    method="pseudobulk",
    config=prop_cfg,
    sample_col=SAMPLE_KEY,
    condition_col=GROUP_KEY,
    celltype_col=composition_key,
    out_dir=str(COMPOSITION_DIR),
    return_type="tuple",
)
prop_df.to_csv(COMPOSITION_DIR / "celltype_proportions_by_sample.csv")
stat_df.to_csv(COMPOSITION_DIR / "celltype_proportion_stats.csv", index=False)
display(prop_df.head())
display(stat_df.head(20))

In [ ]:
sample_conditions = adata.obs[[SAMPLE_KEY, GROUP_KEY]].drop_duplicates().set_index(SAMPLE_KEY)[GROUP_KEY]
sample_conditions = sample_conditions.reindex(prop_df.index)
group_props = prop_df.groupby(sample_conditions.astype(str)).mean()
group_props = group_props.div(group_props.sum(axis=1), axis=0).fillna(0)

scl.al.plot_composition(prop_df, sample_conditions, out_dir=str(FIG_DIR))
try:
    scl.al.plot_grouped_proportion_bar(
        group_props,
        group_order=GROUP_ORDER,
        out_dir=str(FIG_DIR),
    )
except Exception as exc:
    print(f"Grouped proportion plot skipped: {exc}")

## 9. Celltype-Aware PBS vs RT Differential Expression


In [ ]:
de_celltype_key = ANNOTATION_KEY if ANNOTATION_KEY in adata.obs.columns else MARKER_GROUPBY
celltypes = [ct for ct in adata.obs[de_celltype_key].astype(str).unique() if ct and ct != "nan"]
MIN_CELLS_PER_GROUP = 30
celltype_de_results = {}

for ct in sorted(celltypes):
    mask = adata.obs[de_celltype_key].astype(str) == ct
    sub = adata[mask].copy()
    counts = sub.obs[GROUP_KEY].value_counts()
    if counts.get(CONDITION_1, 0) < MIN_CELLS_PER_GROUP or counts.get(CONDITION_2, 0) < MIN_CELLS_PER_GROUP:
        print(f"Skipping {ct}: insufficient cells per group ({counts.to_dict()})")
        continue
    key_added = f"de_{ct}".replace(" ", "_").replace("/", "_")
    cfg = scl.al.CompareGroupsConfig(
        groupby=GROUP_KEY,
        group1=CONDITION_2,
        group2=CONDITION_1,
        method="wilcoxon",
        use_raw=True,
        key_added=key_added,
        n_top_genes=200,
        min_log2fc=0.25,
        max_padj=0.05,
        min_pct=0.1,
    )
    try:
        result = scl.al.compare_groups(sub, config=cfg)
        celltype_de_results[ct] = result
        out_csv = DE_DIR / f"{key_added}_RT_vs_PBS.csv"
        if isinstance(result, pd.DataFrame):
            result.to_csv(out_csv, index=False)
        print(f"Finished {ct}: {sub.n_obs:,} cells")
    except Exception as exc:
        print(f"DE failed for {ct}: {exc}")

adata.uns.setdefault("sclucid", {}).setdefault("analysis", {})["celltype_de_summary"] = {
    "celltype_key": de_celltype_key,
    "condition_key": GROUP_KEY,
    "contrast": f"{CONDITION_2}_vs_{CONDITION_1}",
    "completed_celltypes": list(celltype_de_results.keys()),
}
print(f"Completed DE for {len(celltype_de_results)} cell types")

## 10. Pseudobulk DE Within Cell Types


In [ ]:
try:
    pb_cfg = scl.al.PseudobulkDEConfig(
        sample_col=SAMPLE_KEY,
        condition_key=GROUP_KEY,
        contrasts=[(CONDITION_1, CONDITION_2)],
        groupby=de_celltype_key,
        layer="counts" if "counts" in adata.layers else None,
        use_raw=False,
        min_cells_per_sample=10,
        min_samples_per_condition=2,
        method="auto",
        fallback_to_cell_level=True,
    )
    pb_de = scl.al.run_pseudobulk_de(adata, config=pb_cfg)
    if isinstance(pb_de, pd.DataFrame):
        pb_de.to_csv(DE_DIR / "pseudobulk_celltype_RT_vs_PBS.csv", index=False)
        display(pb_de.head(20))
    else:
        print(type(pb_de))
except Exception as exc:
    print(f"Pseudobulk DE skipped: {exc}")

## 11. Export Metadata & Final Object


In [ ]:
metadata_cols = [c for c in [SAMPLE_KEY, GROUP_KEY, CLUSTER_KEY, MARKER_GROUPBY, ANNOTATION_KEY, MAIN_ANNOTATION_KEY] if c in adata.obs.columns]
metadata = adata.obs[metadata_cols].copy()
metadata.to_csv(RESULTS_DIR / "sce_metadata_LPJ_scLucid.tsv", sep="\t")

summary = {
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "use_rep": USE_REP,
    "umap_key": UMAP_KEY,
    "cluster_key": CLUSTER_KEY,
    "marker_groupby": MARKER_GROUPBY,
    "annotation_key": ANNOTATION_KEY,
    "composition_key": composition_key,
    "condition_key": GROUP_KEY,
    "contrast": f"{CONDITION_2}_vs_{CONDITION_1}",
}
(REVIEW_DIR / "step2_review_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

In [ ]:
adata.write_h5ad(ANNOTATED_H5AD, compression="gzip")
print(f"Saved annotated object: {ANNOTATED_H5AD}")
gc.collect()